# Notebook 3: The Ratchet Learns For You

**Plan A — Automated Search**

You now know what an encoded magic state is (Notebook 1) and how to measure its quality (Notebook 2). This notebook shows how the **autoresearch ratchet** automatically explores the parameter space to find the best circuit configuration.

**What you will learn:**
1. The incumbent-challenger optimization model
2. How challengers are generated (neighbor walk, random combo, lesson-guided)
3. How the ratchet selects winners and extracts lessons
4. Cross-rung propagation and search space narrowing

In [1]:
%matplotlib inline
import sys, warnings, tempfile
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

from autoresearch_quantum.models import (
    ExperimentSpec, RungConfig, EvaluationMetrics,
    QualityWeights, CostWeights, ScoreConfig, SearchSpaceConfig,
    TierPolicyConfig, HardwareConfig, LessonFeedback, SearchRule,
)
from autoresearch_quantum.execution.local import LocalCheapExecutor
from autoresearch_quantum.search.challengers import (
    generate_neighbor_challengers, mutation_summary, GeneratedChallenger,
)
from autoresearch_quantum.search.strategies import (
    NeighborWalk, RandomCombo, LessonGuided, CompositeGenerator,
    default_composite, StrategyWeight,
)
from autoresearch_quantum.ratchet.runner import AutoresearchHarness
from autoresearch_quantum.persistence.store import ResearchStore
from autoresearch_quantum.config import load_rung_config
from autoresearch_quantum.lessons.extractor import extract_rung_lesson
from autoresearch_quantum.lessons.feedback import (
    extract_search_rules, narrow_search_space, build_lesson_feedback,
)
from autoresearch_quantum.execution.transfer import TransferEvaluator

print("All imports successful.")

All imports successful.


In [2]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, order, checkpoint_summary
tracker = LearningTracker("plan_a_03")
print("Learning tracker active.")

Learning tracker active.


---
## 1. The Incumbent-Challenger Model

The ratchet keeps a **best-so-far** configuration called the **incumbent**. Each step:

1. Generate **challengers** — new configurations that differ from the incumbent in one or more parameters
2. Evaluate each challenger on the cheap tier (noisy simulator)
3. If any challenger beats the incumbent by a margin, it becomes the new incumbent
4. Repeat until patience runs out

This is a form of **local search** — like hill climbing in parameter space.

In [3]:
# Load the rung1 configuration
rung_config = load_rung_config("../../configs/rungs/rung1.yaml")

# The bootstrap incumbent
incumbent_spec = rung_config.bootstrap_incumbent
print("Bootstrap incumbent:")
print(f"  seed_style:         {incumbent_spec.seed_style}")
print(f"  encoder_style:      {incumbent_spec.encoder_style}")
print(f"  verification:       {incumbent_spec.verification}")
print(f"  postselection:      {incumbent_spec.postselection}")
print(f"  optimization_level: {incumbent_spec.optimization_level}")
print(f"  target_backend:     {incumbent_spec.target_backend}")
print(f"\nSearch space dimensions:")
for dim, values in rung_config.search_space.dimensions.items():
    print(f"  {dim}: {values}")
print(f"\nMax challengers per step: {rung_config.search_space.max_challengers_per_step}")

Bootstrap incumbent:
  seed_style:         h_p
  encoder_style:      cx_chain
  verification:       both
  postselection:      all_measured
  optimization_level: 2
  target_backend:     fake_brisbane

Search space dimensions:
  seed_style: ['h_p', 'ry_rz', 'u_magic']
  encoder_style: ['cx_chain', 'cz_compiled']
  verification: ['both', 'z_only', 'x_only']
  postselection: ['all_measured', 'z_only', 'none']
  ancilla_strategy: ['dedicated_pair', 'reused_single']
  optimization_level: [1, 2, 3]

Max challengers per step: 8


### The ratchet guarantee

The key property: the incumbent **never gets worse**. A challenger must demonstrably beat the incumbent to replace it.

In [4]:
quiz(tracker, "q1_ratchet_guarantee",
    question="What is the ratchet guarantee?",
    options=[
        "Every step improves the score",
        "The incumbent never gets worse \u2014 challengers must beat it to replace it",
        "The search space shrinks every step",
        "The ratchet always converges to the global optimum",
    ],
    correct=1, section="1. Incumbent-challenger", bloom="remember",
    explanation="The ratchet is monotonic: if no challenger beats the incumbent, the incumbent stays. This does NOT guarantee finding the global optimum.")

---
## 2. Generating Challengers: Neighbor Walk

The simplest strategy: change **one parameter at a time**. For each dimension in the search space, try each alternative value.

In [ ]:
challengers = generate_neighbor_challengers(
    incumbent_spec,
    rung_config.search_space,
)

print(f"Generated {len(challengers)} challengers:\n")
for i, c in enumerate(challengers):
    print(f"  {i+1}. {c.mutation_note}")

In [ ]:
quiz(tracker, "q2_neighborwalk",
    question="How does NeighborWalk generate challengers?",
    options=[
        "Changes all parameters simultaneously to random values",
        "Changes exactly one parameter at a time to each of its other possible values",
        "Applies gradient descent to continuous parameters",
    ],
    correct=1, section="2. Challengers", bloom="understand",
    explanation="NeighborWalk is single-axis: for each dimension, try every alternative value while keeping all other dimensions fixed.")
checkpoint_summary(tracker, "2. Challengers")

> **Key Insight:** Neighbor walk is exhaustive within one axis but never explores *combinations* of changes. It is good for identifying which single parameter matters most.

---
## 3. Evaluating Challengers

Let us run the incumbent and all challengers on the local noisy simulator.

In [ ]:
# Use smaller settings for speed
fast_rung = RungConfig(
    rung=1, name=rung_config.name, description=rung_config.description,
    objective=rung_config.objective, bootstrap_incumbent=incumbent_spec,
    search_space=rung_config.search_space,
    tier_policy=TierPolicyConfig(
        cheap_margin=0.002, cheap_shots=256, cheap_repeats=1,
        expensive_shots=512, expensive_repeats=1,
        promote_top_k=2, enable_hardware=False,
    ),
    score=rung_config.score,
    step_budget=1, patience=1,
    hardware=HardwareConfig(),
)

executor = LocalCheapExecutor()
incumbent_result = executor.evaluate(incumbent_spec, fast_rung)
print(f"Incumbent score: {incumbent_result.score:.4f}")
print(f"  failure_mode: {incumbent_result.metrics.dominant_failure_mode}")

challenger_scores = {}
for c in challengers[:8]:  # Limit for speed
    result = executor.evaluate(c.spec, fast_rung)
    challenger_scores[c.mutation_note] = result.score
    beat = "BEATS" if result.score > incumbent_result.score else "     "
    print(f"  {beat} {c.mutation_note}: {result.score:.4f}")

In [ ]:
# Visualize incumbent vs challengers
fig, ax = plt.subplots(figsize=(12, 5))

labels = ["INCUMBENT"] + list(challenger_scores.keys())
scores = [incumbent_result.score] + list(challenger_scores.values())
colors = ["#e74c3c"] + ["#2ecc71" if s > incumbent_result.score else "#95a5a6" for s in challenger_scores.values()]

bars = ax.barh(range(len(labels)), scores, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels([l[:40] for l in labels], fontsize=8)
ax.set_xlabel("Score")
ax.set_title("Incumbent vs Challengers")
ax.axvline(x=incumbent_result.score, color="#e74c3c", linestyle="--", alpha=0.5, label="Incumbent")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
predict_choice(tracker, "q3_challenger_wins",
    question="Looking at the bar chart: did any challenger beat the incumbent?",
    options=[
        "Yes \u2014 at least one bar is taller than INCUMBENT",
        "No \u2014 the incumbent bar is the tallest",
        "Can't tell from a bar chart",
    ],
    correct=0, section="3. Evaluation", bloom="apply",
    explanation="In most runs, at least one challenger finds a better configuration.")

---
## 4. One Ratchet Step in Slow Motion

Now let the harness run a complete ratchet step — including challenger generation, evaluation, promotion, and winner selection.

In [ ]:
store = ResearchStore(tempfile.mkdtemp())
harness = AutoresearchHarness(store)

step = harness.run_ratchet_step(fast_rung, allow_hardware=False)

print(f"Step index:            {step.step_index}")
print(f"Incumbent before:      {step.incumbent_before_id}")
print(f"Challengers tested:    {len(step.challengers_tested)}")
print(f"Promoted to expensive: {len(step.promoted_challengers)}")
print(f"Winner:                {step.winner_id}")
print(f"Winning margin:        {step.winning_margin:+.4f}")
print(f"\nCheap-tier justification:")
print(f"  {step.cheap_tier_justification}")
print(f"\nDistilled lesson:")
print(f"  {step.distilled_lesson}")

In [ ]:
quiz(tracker, "q4_no_improvement",
    question="What happens if ALL challengers score lower than the incumbent?",
    options=[
        "The harness picks the best challenger anyway",
        "The incumbent stays and the step is logged with zero improvement",
        "The harness generates more challengers until one wins",
    ],
    correct=1, section="4. Ratchet step", bloom="understand",
    explanation="Monotonic guarantee: if no challenger wins, the incumbent stays. Consecutive no-improvement steps trigger patience.")
checkpoint_summary(tracker, "4. Ratchet step")

> **Key Insight:** The  tells you how much the winner improved over the incumbent. If positive, the ratchet "clicked" forward. The  is a human-readable summary of what changed and why.

---
## 5. Running a Full Rung

A **rung** runs multiple ratchet steps in sequence. It stops when the step budget is exhausted or when patience runs out (no improvement for N consecutive steps).

In [ ]:
# Fresh store for a clean rung
store2 = ResearchStore(tempfile.mkdtemp())
harness2 = AutoresearchHarness(store2)

# Run with step_budget=3
full_rung = RungConfig(
    rung=1, name=rung_config.name, description=rung_config.description,
    objective=rung_config.objective, bootstrap_incumbent=incumbent_spec,
    search_space=SearchSpaceConfig(
        dimensions={"verification": ["both", "z_only", "x_only"],
                    "seed_style": ["h_p", "ry_rz", "u_magic"],
                    "postselection": ["all_measured", "z_only", "none"]},
        max_challengers_per_step=6,
    ),
    tier_policy=TierPolicyConfig(
        cheap_margin=0.001, cheap_shots=256, cheap_repeats=1,
        expensive_shots=512, expensive_repeats=1,
        promote_top_k=2, enable_hardware=False,
    ),
    score=rung_config.score,
    step_budget=3, patience=2,
    hardware=HardwareConfig(),
)

steps, lesson, feedback = harness2.run_rung(full_rung, allow_hardware=False)
print(f"Steps completed: {len(steps)}")
for s in steps:
    print(f"  Step {s.step_index}: winner={s.winner_id[:30]}... margin={s.winning_margin:+.4f}")

In [ ]:
# Show the lesson
print("=" * 60)
print(lesson.narrative)
print("=" * 60)

In [ ]:
reflect(tracker, "q5_lesson_quality",
    question="Read the lesson narrative above. What actionable insight does it give? What would make it better?",
    section="5. Lesson", bloom="evaluate",
    model_answer="A good lesson names specific parameter values that helped/hurt and explains WHY. The machine-readable rules are often more actionable than the narrative.")

---
## 6. Visualizing the Exploration

Let us plot how the score evolved and which experiments the ratchet tried.

In [ ]:
experiments = store2.list_experiments(1)
exp_scores = [(e["experiment_id"][:25], e["final_score"], e["role"]) for e in experiments]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#e74c3c" if role == "incumbent" else "#3498db" for _, _, role in exp_scores]
ax.bar(range(len(exp_scores)), [s for _, s, _ in exp_scores], color=colors)
ax.set_xticks(range(len(exp_scores)))
ax.set_xticklabels([eid for eid, _, _ in exp_scores], rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Score")
ax.set_title("All Experiments in Rung 1")

# Add legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#e74c3c", label="Incumbent"), Patch(color="#3498db", label="Challenger")])
plt.tight_layout()
plt.show()

---
## 7. Search Strategies Compared

The harness supports three strategies. Let us see how they generate different challengers from the same incumbent.

In [ ]:
search_space = SearchSpaceConfig(
    dimensions={
        "verification": ["both", "z_only", "x_only"],
        "seed_style": ["h_p", "ry_rz", "u_magic"],
        "optimization_level": [1, 2, 3],
    },
    max_challengers_per_step=6,
)

for name, strategy in [("NeighborWalk", NeighborWalk()),
                        ("RandomCombo", RandomCombo(num_candidates=6)),
                        ("CompositeGenerator", default_composite(has_lessons=False))]:
    challengers = strategy.generate(incumbent_spec, search_space, set())
    print(f"\n{name} ({len(challengers)} challengers):")
    for c in challengers[:4]:
        print(f"  {c.mutation_note}")
    if len(challengers) > 4:
        print(f"  ... and {len(challengers) - 4} more")

### Strategy comparison

- **NeighborWalk**: 1 axis at a time, systematic
- **RandomCombo**: multiple axes, random
- **LessonGuided**: rule-biased from previous rungs

In [ ]:
order(tracker, "q6_strategy_breadth",
    instruction="Rank strategies from narrowest to broadest exploration:",
    items=["NeighborWalk", "RandomCombo", "LessonGuided"],
    correct_order=["NeighborWalk", "LessonGuided", "RandomCombo"],
    section="6. Search strategies", bloom="analyze",
    explanation="NeighborWalk: 1 param (narrowest). LessonGuided: focused by rules (medium). RandomCombo: multiple params randomly (broadest).")

---
## 8. Lesson-Guided Search

After a rung completes, the harness extracts **SearchRules** — machine-readable directives like "prefer z_only" or "avoid x_only". These guide future search.

In [ ]:
# Show the feedback rules from our rung
print(f"Feedback from rung 1: {len(feedback.rules)} rules\n")
for rule in feedback.rules:
    print(f"  {rule.action.upper():7s} {rule.dimension}={rule.value}  (confidence={rule.confidence:.2f})")
    print(f"          {rule.reason}\n")

print(f"Narrowed dimensions:")
for dim, vals in feedback.narrowed_dimensions.items():
    print(f"  {dim}: {vals}")

In [ ]:
quiz(tracker, "q7_fix_vs_avoid",
    question="What is the difference between a 'fix' rule and an 'avoid' rule?",
    options=[
        "'fix' locks a value permanently; 'avoid' removes a value from the search space",
        "'fix' repairs a bug; 'avoid' prevents a crash",
        "They are synonyms",
    ],
    correct=0, section="7. Lesson-guided", bloom="remember",
    explanation="'fix': this value is clearly best, always use it. 'avoid': this value consistently hurts, remove it.")
checkpoint_summary(tracker, "7. Lesson-guided")

In [ ]:
# Run LessonGuided strategy using the feedback
if feedback.rules:
    guided = LessonGuided(num_candidates=6)
    guided_challengers = guided.generate(incumbent_spec, search_space, set(), [feedback])
    print(f"Lesson-guided generated {len(guided_challengers)} challengers:")
    for c in guided_challengers:
        print(f"  {c.mutation_note}")
else:
    print("No rules extracted (too few experiments). Try increasing step_budget.")

---
## 9. Cross-Rung Propagation

A **ratchet** runs multiple rungs in sequence. The winner from rung N becomes the bootstrap incumbent for rung N+1. Lessons narrow the search space.

In [ ]:
store3 = ResearchStore(tempfile.mkdtemp())
harness3 = AutoresearchHarness(store3)

rung1_config = RungConfig(
    rung=1, name="Rung 1", description="Explore seed and verification",
    objective="Find best basic config",
    bootstrap_incumbent=ExperimentSpec(
        rung=1, target_backend="fake_brisbane", noise_backend="fake_brisbane",
        shots=256, repeats=1,
    ),
    search_space=SearchSpaceConfig(
        dimensions={"verification": ["both", "z_only"], "seed_style": ["h_p", "ry_rz"]},
        max_challengers_per_step=4,
    ),
    tier_policy=TierPolicyConfig(cheap_margin=0.0, cheap_shots=256, cheap_repeats=1,
                                  promote_top_k=1, enable_hardware=False),
    score=rung_config.score, step_budget=2, patience=1, hardware=HardwareConfig(),
)

rung2_config = RungConfig(
    rung=2, name="Rung 2", description="Refine with more dimensions",
    objective="Optimize further",
    bootstrap_incumbent=ExperimentSpec(
        rung=2, target_backend="fake_brisbane", noise_backend="fake_brisbane",
        shots=256, repeats=1,
    ),
    search_space=SearchSpaceConfig(
        dimensions={"verification": ["both", "z_only"], "optimization_level": [1, 2, 3]},
        max_challengers_per_step=4,
    ),
    tier_policy=rung1_config.tier_policy,
    score=rung_config.score, step_budget=2, patience=1, hardware=HardwareConfig(),
)

results = harness3.run_ratchet([rung1_config, rung2_config], allow_hardware=False)

for lesson_obj, fb in results:
    print(f"\nRung {lesson_obj.rung}: {lesson_obj.name}")
    print(f"  Rules extracted: {len(fb.rules)}")
    print(f"  Best spec fields: {dict(list(fb.best_spec_fields.items())[:5])}...")

In [ ]:
quiz(tracker, "q8_propagation",
    question="Why does the ratchet propagate the winning spec to the next rung?",
    options=[
        "To save typing the spec again",
        "The winner from rung N is a good starting point for rung N+1, avoiding cold-start",
        "Each rung must use the same spec",
    ],
    correct=1, section="8. Cross-rung", bloom="understand",
    explanation="Cross-rung propagation transfers knowledge: best settings from one rung become the starting point for the next.")

---
## 10. Transfer Evaluation

A **transfer test** runs the best spec across multiple backend noise models to check if the settings generalize or are overfit to one specific noise profile.

In [ ]:
evaluator = TransferEvaluator()
report = evaluator.evaluate_across_backends(
    incumbent_spec,
    ["fake_brisbane"],  # Single backend for speed (add more for real transfer tests)
    fast_rung,
)

print(f"Transfer score (pessimistic = min): {report.transfer_score:.4f}")
print(f"Mean score:  {report.mean_score:.4f}")
print(f"Std score:   {report.std_score:.4f}")
for backend_name, score in report.per_backend_scores.items():
    print(f"  {backend_name}: {score:.4f}")

In [ ]:
quiz(tracker, "q9_transfer_quality",
    question="When is a transfer score 'good'?",
    options=[
        "When it is higher than 0",
        "When it is close to the original score on the source backend",
        "When it is exactly 1.0",
    ],
    correct=1, section="9. Transfer", bloom="evaluate",
    explanation="Good transfer means settings work almost as well on the target backend. A large drop means overfitting to the source noise profile.")
checkpoint_summary(tracker, "9. Transfer")

---
## Summary

You have now seen the complete autoresearch pipeline:

| Layer | What happens |
|---|---|
| **Circuit** | Build encoded magic state (Notebook 1) |
| **Metrics** | Measure quality, cost, and score (Notebook 2) |
| **Search** | Generate and evaluate challengers (this notebook) |
| **Ratchet** | Iterate: incumbent vs challengers, promote winners |
| **Lessons** | Extract rules, narrow search space, propagate to next rung |
| **Transfer** | Verify settings generalize across backends |

The entire process compresses hours of manual parameter exploration into minutes of automated search. Each rung produces a human-readable lesson and machine-readable rules that make future exploration more efficient.

---
## Final Assessment

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")